# Mouse Lung Explorer v3.3

Derive the Full HTML from the pinned v3.2 release. This notebook changes interface layout, DOI summaries, panel exclusivity, motion icons and export composition only; M07 geometry and independent E02 ROI data are unchanged. Run before notebook 19, which derives Lite from this output.


In [1]:
from pathlib import Path
import re
import gzip

root = Path.cwd()
source = root / 'assets/build/viewer-base.html.gz'
output = root / 'viewer/mlung3d_v3.3_full.html'
html = gzip.decompress(source.read_bytes()).decode()

def replace_once(old, new):
    global html
    assert html.count(old) == 1, (old[:90], html.count(old))
    html = html.replace(old, new, 1)

# Preserve the pinned build template; update the public version and current Full download references.
html = html.replace('v3.2', 'v3.3').replace('Explorer 3.2', 'Explorer 3.3')
html = html.replace('Mouse lung explorer · v3.2', 'Mouse lung explorer · v3.3')
html = html.replace('Show CT mouse body', 'Mouse body').replace('Hide CT mouse body', 'Hide body')

old = '''    <div class="btnGrid three compact" aria-label="Viewer tools">
      <button id="toggleClip">Clip</button>
      <button id="resetView" class="ghost">Reset</button>
      <button id="toggleInfo" class="ghost">Info</button>
      <a id="homeLink" class="homeButton" href="https://arashabadi.github.io/" target="_blank" rel="noopener noreferrer" aria-label="Open Arash Bagherabadi homepage">Homepage ↗</a>
    </div>'''
new = '''    <div class="btnGrid two compact" aria-label="Viewer tools">
      <button id="toggleClip">Clip</button>
      <button id="toggleInfo" class="ghost">Info</button>
    </div>
    <div class="btnGrid two" aria-label="Explore and homepage">
      <button id="openAlveolar" type="button" class="primary">Alveolar chicken-wire view</button>
      <a id="homeLink" class="homeButton" href="https://arashabadi.github.io/" target="_blank" rel="noopener noreferrer" aria-label="Open Arash Bagherabadi homepage">Homepage ↗</a>
    </div>'''
replace_once(old, new)
html, count = re.subn(r"document\.getElementById\('resetView'\)\.addEventListener\('click',\(\)=>\{.*?\}\);", '', html, count=1)
assert count == 1
html, count = re.subn(r" const alveolarButton=document\.createElement\('button'\);.*?ui\.insertBefore\(alveolarButton,cuts\);", '', html, count=1)
assert count == 1
replace_once("#ui .homeButton{grid-column:1/-1}", "#ui .homeButton{grid-column:auto}")
replace_once('</style>', '#ui [aria-label="Explore and homepage"]{align-items:stretch}#ui [aria-label="Explore and homepage"]>*{min-height:48px!important;display:flex;align-items:center;justify-content:center;text-align:center}#ui #openAlveolar{font-size:12px!important;line-height:1.25}\n</style>')

old = '<button id="resumeRotateBtn" type="button" class="iconButton" aria-label="Resume automatic rotation" title="Resume or pause automatic rotation"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M20 8a8 8 0 1 0 1 6M20 3v5h-5"/></svg></button>'
new = '<button id="resumeRotateBtn" type="button" class="iconButton" aria-label="Play rotation" title="Play rotation"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M8 5v14l11-7z"/></svg></button>'
replace_once(old,new)
old = "function setResumeButton(){resumeRotateBtn.setAttribute('aria-pressed',String(!autoRotateStoppedByUser));resumeRotateBtn.title=autoRotateStoppedByUser?'Resume automatic rotation':'Pause automatic rotation';}"
new = "function setResumeButton(){const paused=autoRotateStoppedByUser;resumeRotateBtn.setAttribute('aria-pressed',String(!paused));resumeRotateBtn.setAttribute('aria-label',paused?'Play rotation':'Pause rotation');resumeRotateBtn.title=paused?'Play rotation':'Pause rotation';resumeRotateBtn.querySelector('svg').innerHTML=paused?'<path d=\"M8 5v14l11-7z\"/>':'<rect x=\"6\" y=\"5\" width=\"4\" height=\"14\" rx=\"1\"/><rect x=\"14\" y=\"5\" width=\"4\" height=\"14\" rx=\"1\"/>'; }"
replace_once(old,new)

# Make Info a compact provenance guide with DOI links, retaining live plane IDs.
start = html.index('<aside id="infoPanel"')
end = html.index('</aside>', start) + len('</aside>')
html = html[:start] + '''<aside id="infoPanel" class="panel" hidden aria-hidden="true">
  <button id="infoClose" type="button" class="ghost">Close</button>
  <div class="eyebrow">Info</div><h2>Model & sources</h2>
  <p class="small">M07 is a reference lung, not a registration of either Xenium section. The whole-lung airspaces are schematic; the E02 alveolar ROI is a separate measured specimen.</p>
  <div class="planeCard"><div class="planeKicker">Current plane · M07 coordinates (mm)</div>
    <div id="planeNavBase" class="planeLine"></div><div id="planeNormal" class="planeMono"></div><div id="planePoint" class="planeMono"></div></div>
  <div class="planeCard"><div class="planeKicker">Sources</div><ul class="citationList">
    <li><a href="https://doi.org/10.1152/japplphysiol.00615.2019" target="_blank" rel="noopener">Bauer et al. (2020)</a> — M07 lung anatomy.</li>
    <li><a href="https://doi.org/10.25820/9arg-9w56" target="_blank" rel="noopener">Beichel et al.</a> — source geometry archive.</li>
    <li><a href="https://doi.org/10.1038/s41592-021-01103-9" target="_blank" rel="noopener">Bolaños et al. (2021)</a> — illustrative whole-body CT context.</li>
    <li><a href="https://doi.org/10.1007/s00335-025-10156-6" target="_blank" rel="noopener">Ruberte et al. (2025)</a> — quadruped orientation terms.</li>
    <li><a href="https://doi.org/10.1371/journal.pone.0183979" target="_blank" rel="noopener">Lovric et al. (2017)</a> — independent alveolar ROI.</li>
  </ul></div>
  <div class="planeCard"><div class="planeKicker">How to cite</div><p class="small">Arash Bagherabadi and Abel Lopez. <em>Mouse Lung Explorer</em> (v3.3) [Software]. <a href="https://arashabadi.github.io/mlung3d/" target="_blank" rel="noopener">arashabadi.github.io/mlung3d/</a></p></div>
  <div class="planeCard"><div class="planeKicker">Acknowledgements</div><p class="small">Thank you to <a href="https://github.com/alopez789" target="_blank" rel="noopener">Abel Lopez</a> for helping build the 3D lung and airway model, and to Codex (GPT-6) for assistance with implementation and documentation.</p></div>
</aside>''' + html[end:]

# Only one floating viewer panel may be open. Event capture runs before module listeners.
script = '''<script>
document.addEventListener('click',event=>{
  const id=event.target.closest('button')?.id;
  if(!['toggleInfo','openAlveolar','saveLungPng','recordLungGif','recordLungVideo'].includes(id))return;
  for(const panelId of ['pngSetup','gifSetup','videoSetup']){
    if((id==='saveLungPng'&&panelId==='pngSetup')||(id==='recordLungGif'&&panelId==='gifSetup')||(id==='recordLungVideo'&&panelId==='videoSetup'))continue;
    const panel=document.getElementById(panelId);if(panel)panel.hidden=true;
  }
  if(id!=='toggleInfo'){
    const info=document.getElementById('infoPanel');if(info){info.hidden=true;info.classList.remove('open');info.setAttribute('aria-hidden','true');}
  }
},true);
</script>'''
replace_once('</body>',script+'</body>')

# The 3D/2D export gets separate, non-overlapping title, image, address and credit zones.
replace_once('y=h-size-100,scale=size/128', 'y=h-size-170,scale=size/128')
replace_once("ctx.fillText('View: '+(guide.dataset.viewName||'Ventral'),x,y+size+19,size+20);ctx.fillText('Mouse left: screen '+(guide.dataset.mouseLeft||'right'),x,y+size+39,size+20);ctx.fillText('Cranial: screen '+(guide.dataset.cranial||'top'),x,y+size+59,size+20);", "ctx.fillText('View: '+(guide.dataset.viewName||'Ventral'),x,y+size+30,size+90);ctx.fillText('Mouse left: screen '+(guide.dataset.mouseLeft||'right'),x,y+size+70,size+90);ctx.fillText('Cranial: screen '+(guide.dataset.cranial||'top'),x,y+size+110,size+90);")
old = """      const pad=Math.max(10,Math.round(w*.014)),textX=panelX+pad,innerW=boxW-2*pad;
      ctx.fillStyle='#d9f4fd';ctx.font='700 '+Math.max(14,Math.round(w/95))+'px system-ui,sans-serif';ctx.fillText('M07 MODEL SECTION',textX,gap+pad+3,innerW);
      ctx.fillStyle='#bfd4dd';ctx.font=Math.max(12,Math.round(w/112))+'px system-ui,sans-serif';ctx.fillText(document.getElementById('planeName').textContent,textX,gap+pad+25,innerW);
      const addressY=h-gap-Math.max(105,Math.round(h*.19));
      const section=document.getElementById('sliceCanvas'),imageTop=gap+pad+36,imageH=Math.max(30,Math.min(innerW*section.height/section.width,addressY-imageTop-15)),imageW=Math.min(innerW,imageH*section.width/section.height);
      ctx.drawImage(transparent?darkPixelsToAlpha(section):section,textX+(innerW-imageW)/2,imageTop+(addressY-12-imageTop-imageH)/2,imageW,imageH);
      ctx.strokeStyle='rgba(160,204,220,.3)';ctx.beginPath();ctx.moveTo(textX,addressY-13);ctx.lineTo(textX+innerW,addressY-13);ctx.stroke();
      ctx.fillStyle='#b5e9f5';ctx.font='700 '+Math.max(11,Math.round(w/113))+'px system-ui,sans-serif';ctx.fillText('REOPEN THIS PLANE',textX,addressY+4,innerW);
      const planeID=encodePlane(sectionAngle,sectionTilt,sectionDepth),breakAt=planeID.indexOf(';');
      ctx.fillStyle='#dcecf2';ctx.font=Math.max(10,Math.round(w/120))+'px ui-monospace,SFMono-Regular,monospace';
      ctx.fillText(planeID.slice(0,breakAt+1),textX,addressY+25,innerW);
      ctx.fillText(planeID.slice(breakAt+1),textX,addressY+46,innerW);
      ctx.fillStyle='#9cb6c4';ctx.font=Math.max(10,Math.round(w/130))+'px system-ui,sans-serif';
      ctx.fillText('az / el: tilt in degrees · d: offset in mm',textX,addressY+66,innerW);"""
new = """      const pad=Math.max(10,Math.round(w*.014)),textX=panelX+pad,innerW=boxW-2*pad,unit=w/1600;
      ctx.textAlign='left';ctx.fillStyle='#d9f4fd';ctx.font='700 '+Math.round(24*unit)+'px system-ui,sans-serif';ctx.fillText('M07 MODEL SECTION',textX,gap+pad+32*unit,innerW);
      ctx.fillStyle='#bfd4dd';ctx.font=Math.round(17*unit)+'px system-ui,sans-serif';ctx.fillText(document.getElementById('planeName').textContent,textX,gap+pad+62*unit,innerW);
      const addressY=h-Math.max(230,Math.round(h*.19));
      const section=document.getElementById('sliceCanvas'),imageTop=gap+pad+85*unit,imageH=Math.max(30,Math.min(innerW*section.height/section.width,addressY-imageTop-28*unit)),imageW=Math.min(innerW,imageH*section.width/section.height);
      ctx.drawImage(transparent?darkPixelsToAlpha(section):section,textX+(innerW-imageW)/2,imageTop+(addressY-22*unit-imageTop-imageH)/2,imageW,imageH);
      ctx.strokeStyle='rgba(160,204,220,.3)';ctx.beginPath();ctx.moveTo(textX,addressY-26*unit);ctx.lineTo(textX+innerW,addressY-26*unit);ctx.stroke();
      ctx.fillStyle='#b5e9f5';ctx.font='700 '+Math.round(16*unit)+'px system-ui,sans-serif';ctx.fillText('REOPEN THIS PLANE',textX,addressY+8*unit,innerW);
      const planeID=encodePlane(sectionAngle,sectionTilt,sectionDepth),breakAt=planeID.indexOf(';');
      ctx.fillStyle='#dcecf2';ctx.font=Math.round(16*unit)+'px ui-monospace,SFMono-Regular,monospace';
      ctx.fillText(planeID.slice(0,breakAt+1),textX,addressY+36*unit,innerW);
      ctx.fillText(planeID.slice(breakAt+1),textX,addressY+64*unit,innerW);
      ctx.fillStyle='#9cb6c4';ctx.font=Math.round(14*unit)+'px system-ui,sans-serif';
      ctx.fillText('az / el: tilt in degrees · d: offset in mm',textX,addressY+91*unit,innerW);"""
replace_once(old,new)

output.write_text(html)
assert output.stat().st_size > 40_000_000
print('Wrote', output, output.stat().st_size)


Wrote /Users/ashi/github/mlung3d/viewer/mlung3d_v3.3_full.html 44937783


## Export color key (in-place v3.3 revision)

The on-screen legend is the sole source for the six anatomical names and swatch colors. The shared `presentationFrame` canvas is used by the JPEG/transparent PNG, GIF and supplementary video paths; its color key is therefore identical across media. The separate E02 ROI exports intentionally have no M07 lobe key. This cell edits the existing v3.3 Full viewer and does not create a new version or change anatomical geometry.

In [2]:
from pathlib import Path
import re, subprocess, hashlib
root=Path.cwd()
viewer=root/'viewer/mlung3d_v3.3_full.html'
html=viewer.read_text()
assert html.count('function presentationFrame(')==1
assert html.count('function drawExportLobeLegend(')==0
assert html.count('presentationFrame(')==5  # definition, JPEG/PNG, GIF, video tick, video first frame
assert html.count('<div class="legendGrid">')==1

legend_js="""  // Read names and swatches from the on-screen Display legend, avoiding duplicate color maps.
  function drawExportLobeLegend(ctx,width,height,sceneWidth){
    const items=[...document.querySelectorAll('.legendGrid .legendItem')].map(node=>({
      name:node.textContent.trim(),color:getComputedStyle(node.querySelector('.dot')).backgroundColor
    }));
    if(items.length!==6)throw new Error('Export lobe key must match the six Display legend entries');
    const scale=Math.max(.65,Math.min(1.8,width/1280));
    const margin=Math.max(12,Math.round(width*.018));
    const panelWidth=Math.round(Math.min(sceneWidth*.46,310*scale));
    const pad=Math.round(12*scale),titleSize=Math.round(13*scale),bodySize=Math.round(13*scale);
    const rowHeight=Math.round(24*scale),noteSize=Math.round(10*scale);
    const note=wholeAlveolarGroup.visible?'Airspaces shown: schematic, not measured':'';
    const panelHeight=pad*2+titleSize+Math.round(10*scale)+items.length*rowHeight+(note?noteSize+Math.round(12*scale):0);
    const x=sceneWidth<width?sceneWidth-panelWidth-margin:margin;
    const y=Math.max(44,Math.round(height*.095));
    ctx.save();ctx.fillStyle='rgba(5,14,21,.84)';ctx.fillRect(x,y,panelWidth,panelHeight);
    ctx.strokeStyle='rgba(179,215,226,.45)';ctx.lineWidth=Math.max(1,scale);ctx.strokeRect(x+.5,y+.5,panelWidth-1,panelHeight-1);
    ctx.textAlign='left';ctx.textBaseline='middle';ctx.fillStyle='#e8f8fd';
    ctx.font='700 '+titleSize+'px system-ui,sans-serif';ctx.fillText('LOBE COLOR KEY',x+pad,y+pad+titleSize/2,panelWidth-2*pad);
    const rowTop=y+pad+titleSize+Math.round(10*scale);
    items.forEach((item,i)=>{
      const cy=rowTop+i*rowHeight+rowHeight/2;
      ctx.fillStyle=item.color;ctx.beginPath();ctx.arc(x+pad+Math.round(6*scale),cy,Math.max(4,Math.round(5*scale)),0,Math.PI*2);ctx.fill();
      ctx.fillStyle='#e8f2f6';ctx.font='600 '+bodySize+'px system-ui,sans-serif';
      ctx.fillText(item.name,x+pad+Math.round(18*scale),cy,panelWidth-2*pad-Math.round(18*scale));
    });
    if(note){ctx.fillStyle='#b9d3dd';ctx.font=noteSize+'px system-ui,sans-serif';ctx.fillText(note,x+pad,y+panelHeight-pad-noteSize/2,panelWidth-2*pad)}
    ctx.restore();
  }
"""
html=html.replace('  function presentationFrame(maxWidth,forGif=false,showSection=false,transparent=false){',legend_js+'  function presentationFrame(maxWidth,forGif=false,showSection=false,transparent=false){',1)
old="""    if(wholeAlveolarGroup.visible){ctx.save();ctx.fillStyle='#d7f3ff';ctx.shadowColor='#061018';ctx.shadowBlur=5;ctx.font=Math.max(9,Math.round(w/110))+'px system-ui,sans-serif';ctx.fillText('SCHEMATIC AIRSPACES · ILLUSTRATIVE',Math.max(12,w*.016),Math.max(18,h*.04));ctx.restore()}\n"""
assert html.count(old)==1
html=html.replace(old,'',1)
anchor='    scene.background=oldBackground;renderer.render(scene,camera);return out;'
assert html.count(anchor)==1
html=html.replace(anchor,"    drawExportLobeLegend(ctx,w,h,forGif&&showSection?Math.round(w*.63)-Math.max(12,Math.round(w*.015)):w);\n"+anchor,1)
viewer.write_text(html)
# Check the exact current Full viewer, including all three export call sites and the six visible swatches.
main=re.search(r'<script type="module">([\s\S]*?)</script>',html)
assert main,'Module script not found'
syntax=subprocess.run(['node','--input-type=module','--check'],input=main.group(1),text=True,capture_output=True)
assert syntax.returncode==0,syntax.stderr
assert html.count('drawExportLobeLegend(ctx,w,h,forGif&&showSection?Math.round(w*.63)-Math.max(12,Math.round(w*.015)):w)')==1
assert html.count('presentationFrame(')==5
assert all(html.count(f'background:{color}')>=1 for color in ['#b83d47','#297d94','#e69130','#479466','#875eab','#ded6ba'])
print('Updated existing Full viewer:',viewer)
print('SHA-256:',hashlib.sha256(viewer.read_bytes()).hexdigest())
print('Checks: JS module syntax; one shared legend call; 5 presentation-frame references; six Display swatches.')

Updated existing Full viewer: /Users/ashi/github/mlung3d/viewer/mlung3d_v3.3_full.html
SHA-256: 84989a25938fe6b7dacd2123a438c4edb2ec684afbf3479f0386e244ff276409
Checks: JS module syntax; one shared legend call; 5 presentation-frame references; six Display swatches.


## Expanded export orientation key

The exported compass retains the colored axis initials and adds the full anatomical terms beneath it. Its position and text size adapt to the output canvas so the heading and key remain inside the frame.

In [3]:
from pathlib import Path
import re, subprocess, hashlib
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html'
html=viewer.read_text()
old=re.search(r'  function drawCompactCompass\(ctx,w,h,onLeft=false\)\{[\s\S]*?\n  \}\n  function darkPixelsToAlpha',html)
assert old, 'Compact export compass not found'
new="""  function drawCompactCompass(ctx,w,h,onLeft=false){
    const svg=document.getElementById('orientationSvg');
    const size=Math.max(100,Math.min(240,w*.16,h*.28));
    const fontSize=Math.max(12,Math.min(18,Math.round(h/85)));
    const lineHeight=Math.round(fontSize*1.4);
    const x=Math.max(36,Math.round(w*.025));
    const y=h-size-(6*lineHeight+105),scale=size/128;
    ctx.save();ctx.translate(x,y);ctx.scale(scale,scale);
    for(const line of svg.querySelectorAll('line')){ctx.beginPath();ctx.moveTo(Number(line.getAttribute('x1')),Number(line.getAttribute('y1')));ctx.lineTo(Number(line.getAttribute('x2')),Number(line.getAttribute('y2')));ctx.strokeStyle=line.getAttribute('stroke')||'#d7f3ff';ctx.globalAlpha=Number(line.getAttribute('opacity')||1);ctx.lineWidth=2.5;ctx.lineCap='round';ctx.stroke()}
    ctx.globalAlpha=1;ctx.fillStyle='#d7f3ff';ctx.beginPath();ctx.arc(64,64,3,0,Math.PI*2);ctx.fill();ctx.font='bold 10px system-ui,sans-serif';ctx.textAlign='center';ctx.textBaseline='middle';
    for(const label of svg.querySelectorAll('text')){ctx.fillStyle=label.getAttribute('fill')||'#d7f3ff';ctx.fillText(label.textContent,Number(label.getAttribute('x')),Number(label.getAttribute('y')))}ctx.restore();
    const guide=document.getElementById('orientationText');
    const lines=[
      'View: '+(guide.dataset.viewName||'Ventral'),
      'Mouse left: screen '+(guide.dataset.mouseLeft||'right'),
      'Cranial: screen '+(guide.dataset.cranial||'top'),
      'R = Right   ·   L = Left',
      'V = Ventral   ·   D = Dorsal',
      'Cr = Cranial   ·   Cd = Caudal'
    ];
    const maxWidth=Math.min(390,w-x-24);
    ctx.save();ctx.textAlign='left';ctx.textBaseline='middle';ctx.shadowColor='#061019';ctx.shadowBlur=5;
    ctx.fillStyle='#d7f3ff';ctx.font='700 '+fontSize+'px system-ui,sans-serif';ctx.fillText('ANATOMICAL ORIENTATION',x,y-18,maxWidth);
    ctx.font='600 '+fontSize+'px system-ui,sans-serif';
    lines.forEach((line,i)=>{ctx.fillStyle=i<3?'#e2f0f6':'#bdd8e2';ctx.fillText(line,x,y+size+22+i*lineHeight,maxWidth)});
    ctx.restore();
  }
  function darkPixelsToAlpha"""
html=html[:old.start()]+new+html[old.end():]
# Keep creator credit in the lower-right corner in both main and independent ROI exports.
old_credit="cx=forGif&&showSection?w-Math.max(18,Math.round(w*.02)):w/2,cy=h-Math.max(11,Math.round(h*.02))"
assert html.count(old_credit)==1
html=html.replace(old_credit,"cx=w-Math.max(18,Math.round(w*.02)),cy=h-Math.max(11,Math.round(h*.02))",1)
assert html.count("ctx.textAlign=forGif&&showSection?'right':'center'")==1
html=html.replace("ctx.textAlign=forGif&&showSection?'right':'center'","ctx.textAlign='right'",1)
assert html.count("ctx.fillRect(cx-tw/2-8,cy-fs-4,tw+16,fs+11)")==1
html=html.replace("ctx.fillRect(cx-tw/2-8,cy-fs-4,tw+16,fs+11)","ctx.fillRect(cx-tw-8,cy-fs-4,tw+16,fs+11)",1)
roi_credit="ctx.fillText('Created by Arash · arashabadi.github.io/mlung3d',pad,y+Math.round(width*.071));"
assert html.count(roi_credit)==1
html=html.replace(roi_credit,"ctx.save();ctx.textAlign='right';ctx.fillText('Created by Arash · arashabadi.github.io/mlung3d',width-pad,height-pad);ctx.restore();",1)
viewer.write_text(html)
module=re.search(r'<script type="module">([\s\S]*?)</script>',html)
assert module
check=subprocess.run(['node','--input-type=module','--check'],input=module.group(1),text=True,capture_output=True)
assert check.returncode==0,check.stderr
for label in ['R = Right','V = Ventral','Cr = Cranial']:
    assert label in html
print('Updated export compass:',viewer)
print('SHA-256:',hashlib.sha256(viewer.read_bytes()).hexdigest())


Updated export compass: /Users/ashi/github/mlung3d/viewer/mlung3d_v3.3_full.html
SHA-256: 1204adb5b678934fbdfbdbc0d4dafcdfb6421fbd502e82b80be3f392aa9931b4


## Consistent compact export layout

The six M07 colors use the same source as the on-screen legend. A small unboxed key is placed at the top right of the visual field: within the section image when clipped, and within the model view otherwise. The independent E02 alveolar ROI keeps the same split-panel and corner-credit style but does not receive an M07 lobe key because it is a different specimen.

In [4]:
from pathlib import Path
import re, subprocess, hashlib
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html'
html=viewer.read_text()
old=re.search(r'  function drawExportLobeLegend\(ctx,width,height,sceneWidth\)\{[\s\S]*?\n  \}\n  function presentationFrame',html)
assert old,'Export color key not found'
new="""  function drawExportLobeLegend(ctx,width,height,clipped){
    // A compact key sits inside the visual field, with no floating panel or border.
    const items=[...document.querySelectorAll('.legendGrid .legendItem')].map(node=>({
      name:node.textContent.trim(),color:getComputedStyle(node.querySelector('.dot')).backgroundColor
    }));
    if(items.length!==6)throw new Error('Export lobe key must match the six Display legend entries');
    const scale=Math.max(.7,Math.min(1.5,width/1600));
    const margin=Math.max(16,Math.round(width*.024));
    const keyWidth=Math.round(184*scale),fontSize=Math.round(12*scale),rowHeight=Math.round(18*scale);
    const x=width-margin-keyWidth;
    const y=clipped?height-Math.max(230,Math.round(height*.19))+Math.round(4*scale):Math.max(62,Math.round(height*.10));
    ctx.save();ctx.textAlign='left';ctx.textBaseline='middle';ctx.shadowColor='rgba(0,0,0,.95)';ctx.shadowBlur=Math.max(3,Math.round(5*scale));
    ctx.font='700 '+fontSize+'px system-ui,sans-serif';ctx.fillStyle='#e8f8fd';ctx.fillText('LOBE COLORS',x,y,keyWidth);
    items.forEach((item,i)=>{
      const cy=y+Math.round(21*scale)+(i+.5)*rowHeight;
      ctx.fillStyle=item.color;ctx.beginPath();ctx.arc(x+Math.round(5*scale),cy,Math.max(3,Math.round(4*scale)),0,Math.PI*2);ctx.fill();
      ctx.fillStyle='#e8f2f6';ctx.font='600 '+fontSize+'px system-ui,sans-serif';ctx.fillText(item.name,x+Math.round(16*scale),cy,keyWidth-Math.round(16*scale));
    });
    if(wholeAlveolarGroup.visible){ctx.fillStyle='#b9d3dd';ctx.font=Math.round(10*scale)+'px system-ui,sans-serif';ctx.fillText('Airspaces: schematic',x,y+Math.round(28*scale)+items.length*rowHeight,keyWidth)}
    ctx.restore();
  }
  function presentationFrame"""
html=html[:old.start()]+new+html[old.end():]
old_call='drawExportLobeLegend(ctx,w,h,forGif&&showSection?Math.round(w*.63)-Math.max(12,Math.round(w*.015)):w);'
assert html.count(old_call)==1
html=html.replace(old_call,'drawExportLobeLegend(ctx,w,h,forGif&&showSection);',1)
viewer.write_text(html)
module=re.search(r'<script type="module">([\s\S]*?)</script>',html)
assert module
result=subprocess.run(['node','--input-type=module','--check'],input=module.group(1),text=True,capture_output=True)
assert result.returncode==0,result.stderr
assert 'strokeRect(x+.5' not in html
assert html.count('drawExportLobeLegend(ctx,w,h,forGif&&showSection)')==1
print('Updated compact, unboxed export key:',viewer)
print('SHA-256:',hashlib.sha256(viewer.read_bytes()).hexdigest())


Updated compact, unboxed export key: /Users/ashi/github/mlung3d/viewer/mlung3d_v3.3_full.html
SHA-256: 1daaa96e369a44ad38fcc84c45377fe9b3d029efdc4c7d65827b90ab336600d9


## Layout review: reserved color-key band
The section image, two-column color key, plane address and footer use separate vertical bands at all export presets. This prevents the color key from overlapping either the tissue image or the address.

In [5]:
from pathlib import Path
import re, subprocess
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html'
html=viewer.read_text()
start=html.index('  function drawExportLobeLegend(')
end=html.index('  function presentationFrame(',start)
html=html[:start]+"""  function drawExportLobeLegend(ctx,width,height,clipped){
    const items=[...document.querySelectorAll('.legendGrid .legendItem')].map(node=>({name:node.textContent.trim(),color:getComputedStyle(node.querySelector('.dot')).backgroundColor}));
    const unit=width/1600, gap=width*.015, pad=width*.014;
    const x=clipped?Math.round(width*.63)+pad:width-width*.024-280*unit;
    const areaWidth=clipped?width-Math.round(width*.63)-gap-2*pad:280*unit;
    const addressY=height-height*.19;
    const y=clipped?addressY-92*unit:height*.10;
    const row=22*unit,font=12*unit;
    ctx.save();ctx.textAlign='left';ctx.textBaseline='middle';ctx.font='600 '+font+'px system-ui,sans-serif';
    items.forEach((item,i)=>{
      const column=clipped?i%2:0,r=clipped?Math.floor(i/2):i;
      const px=x+column*areaWidth/2,py=y+r*row;
      ctx.fillStyle=item.color;ctx.beginPath();ctx.arc(px+4*unit,py,3*unit,0,Math.PI*2);ctx.fill();
      ctx.fillStyle='#dcecf2';ctx.fillText(item.name,px+14*unit,py,(clipped?areaWidth/2:areaWidth)-20*unit);
    });
    if(wholeAlveolarGroup.visible){ctx.fillStyle='#b9d3dd';ctx.font=10*unit+'px system-ui,sans-serif';ctx.fillText('Airspaces: schematic',x,y+(clipped?3:6)*row,areaWidth)}
    ctx.restore();
  }
"""+html[end:]
html=html.replace('const addressY=h-Math.max(230,Math.round(h*.19));','const addressY=h-h*.19;')
html=html.replace('addressY-imageTop-28*unit','addressY-imageTop-125*unit').replace('addressY-22*unit-imageTop-imageH','addressY-119*unit-imageTop-imageH')
viewer.write_text(html)
module=re.search(r'<script type="module">([\s\S]*?)</script>',html).group(1)
r=subprocess.run(['node','--input-type=module','--check'],input=module,text=True,capture_output=True)
assert r.returncode==0,r.stderr
# Check independent section, legend, address and footer bands at every preset.
for width in [640,960,1280,1920,3200]:
    height=width*9/16;unit=width/1600;address=height*.81
    image_bottom=address-119*unit;legend_top=address-92*unit-6*unit
    legend_bottom=address-92*unit+66*unit+5*unit
    assert image_bottom<legend_top and legend_bottom<address
    assert address+91*unit<height*.98-30*unit
print('Section / legend / address / credit bands are disjoint at all five export widths.')


Section / legend / address / credit bands are disjoint at all five export widths.


## Review of independent export layouts
Keep the accepted clipped layout. Unclipped exports put the plane address beneath the title. E02 ROI image and video retain two visual columns, preserve source aspect ratio, and separate source citation from the creator footer. The ROI has no independent GIF export.

In [6]:
from pathlib import Path
import re, subprocess
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html'
html=viewer.read_text()
# Preserve the accepted clipped composition; only change the unclipped address.
old="if(sectionPlane.visible){ctx.save();ctx.font=Math.max(10,Math.round(w/75))+'px system-ui,sans-serif';ctx.fillStyle='#d7f3ff';ctx.shadowColor='#061018';ctx.shadowBlur=5;const line=encodePlane(sectionAngle,sectionTilt,sectionDepth);ctx.fillText(line,Math.max(12,w*.02),h-Math.max(14,h*.04));ctx.restore()}"
new="if(sectionPlane.visible){ctx.save();ctx.textAlign='left';ctx.font=Math.max(8,Math.round(w/120))+'px ui-monospace,monospace';ctx.fillStyle='#bdd8e2';const line=encodePlane(sectionAngle,sectionTilt,sectionDepth),split=line.indexOf(';'),x=w*.02;ctx.fillText(line.slice(0,split+1),x,h*.085,w*.65);ctx.fillText(line.slice(split+1),x,h*.115,w*.65);ctx.restore()}"
assert html.count(old)==1
html=html.replace(old,new,1)
old='ctx.drawImage(source,0,0,source.width,source.height,pad,top,leftW-pad*2,contentH);'
new='const fit=Math.min((leftW-pad*2)/source.width,contentH/source.height),dw=source.width*fit,dh=source.height*fit;ctx.drawImage(source,0,0,source.width,source.height,pad+(leftW-pad*2-dw)/2,top+(contentH-dh)/2,dw,dh);'
assert html.count(old)==1
html=html.replace(old,new,1)
html=html.replace("ctx.font='700 '+Math.round(width*.025)+'px system-ui,sans-serif';ctx.fillText('CHICKEN-WIRE VIEW · E02 ROI'", "ctx.font='700 '+Math.round(width*.015)+'px system-ui,sans-serif';ctx.fillText('CHICKEN-WIRE VIEW · E02 ROI · v3.3'",1)
html=html.replace('const y=Math.round(height*.82);','const y=Math.round(height*.81);',1)
html=html.replace('y+Math.round(width*.023)','y+Math.round(width*.020)',1)
html=html.replace('y+Math.round(width*.041)','y+Math.round(width*.037)',1)
html=html.replace('y+Math.round(width*.056)','y+Math.round(width*.053)',1)
html=html.replace("ctx.save();ctx.textAlign='right';ctx.fillText('Created by Arash · arashabadi.github.io/mlung3d',width-pad,height-pad);ctx.restore();", "ctx.save();ctx.textAlign='right';ctx.font='600 '+Math.max(14,Math.round(width/105))+'px system-ui,sans-serif';ctx.fillText('Created by Arash · arashabadi.github.io/mlung3d',width-width*.02,height-height*.02);ctx.restore();",1)
viewer.write_text(html)
module=re.search(r'<script type="module">([\s\S]*?)</script>',html).group(1)
r=subprocess.run(['node','--input-type=module','--check'],input=module,text=True,capture_output=True)
assert r.returncode==0,r.stderr
for width in [1280,1920,3200]:
    height=width*9/16
    citation_bottom=height*.81+width*.053
    credit_top=height*.98-max(14,round(width/105))
    assert citation_bottom<credit_top
for sw,sh in [(1600,900),(900,1600),(1000,1000)]:
    fit=min(730/sw,470/sh)
    assert abs((sw*fit)/(sh*fit)-sw/sh)<1e-10
print('Clip preserved; unclipped address separated; ROI aspect ratio and footer spacing checked.')


Clip preserved; unclipped address separated; ROI aspect ratio and footer spacing checked.


## Website-only export attribution
All current export composers use the exact credit: Created by arashabadi.github.io/mlung3d. The full author name remains in the Info citation.

In [7]:
from pathlib import Path
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html'
html=viewer.read_text()
old='Created by Arash · arashabadi.github.io/mlung3d'
assert html.count(old)==2, 'Expected main and independent ROI export credits'
html=html.replace(old,'Created by arashabadi.github.io/mlung3d')
assert 'Created by Arash' not in html
viewer.write_text(html)
print('Updated shared image/GIF/video and ROI image/video credits.')


Updated shared image/GIF/video and ROI image/video credits.


## Live Clip changes during recording
GIF and video keep their fixed canvas size while switching between the existing clipped and unclipped layouts. Entering Clip refreshes the section before the next recorded frame. Recording continues without restarting the encoder.

In [8]:
from pathlib import Path
import re, subprocess
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html'
html=viewer.read_text()
assert html.count("document.getElementById('toggleClip').disabled=true;")==2
html=html.replace("document.getElementById('toggleClip').disabled=true;",'')
# Refresh the section immediately when entering Clip, before composing the next frame.
html=html.replace('const frame=presentationFrame(gifOptions.width,true,gifLayoutClipped,gifTransparent)', 'if(clipping&&!gifLayoutClipped)drawSlice();gifLayoutClipped=clipping;\n        const frame=presentationFrame(gifOptions.width,true,gifLayoutClipped,gifTransparent)',1)
html=html.replace('try{const frame=presentationFrame(videoCanvas.width,true,videoClip,videoTransparent)', 'try{if(clipping&&!videoClip)drawSlice();videoClip=clipping;const frame=presentationFrame(videoCanvas.width,true,videoClip,videoTransparent)',1)
viewer.write_text(html)
module=re.search(r'<script type="module">([\s\S]*?)</script>',html).group(1)
r=subprocess.run(['node','--input-type=module','--check'],input=module,text=True,capture_output=True)
assert r.returncode==0,r.stderr
assert "document.getElementById('toggleClip').disabled=true" not in html
print('GIF and video now use live Clip state with fixed 16:9 frame dimensions.')


GIF and video now use live Clip state with fixed 16:9 frame dimensions.


## Consistent orientation compass
Separate projected direction labels with a bounded candidate search. Use cyan, amber and lavender for the three axes. SVG labels and export canvas labels use identical positions and colors, with a dark text halo for legibility.

In [9]:
from pathlib import Path
import re, subprocess
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html'
html=viewer.read_text()
a=html.index('    <svg id="orientationSvg"');b=html.index('</svg>',a)+6
svg=html[a:b]
for old,new in {'#83d7ff':'#67c5e8','#9bdfff':'#67c5e8','#ffc38d':'#e9b872','#ffd1a4':'#e9b872','#8effb4':'#b5a1e5','#a9efb9':'#b5a1e5'}.items():svg=svg.replace(old,new)
svg=svg.replace('font-size="11"','font-size="10" font-weight="600" font-family="system-ui, sans-serif" stroke="#080d12" stroke-width="3" paint-order="stroke fill" stroke-linejoin="round"')
html=html[:a]+svg+html[b:]
html=html.replace('Blue: left / right &nbsp;·&nbsp; Amber: ventral / dorsal &nbsp;·&nbsp; Green: cranial / caudal','<span style="color:#67c5e8">R / L · Right / Left</span><br><span style="color:#e9b872">V / D · Ventral / Dorsal</span><br><span style="color:#b5a1e5">Cr / Cd · Cranial / Caudal</span>')
anchor='    const leftSide=screenSideName('
assert html.count(anchor)==1
layout="""    // Place labels outside the axis tips and separate projected directions.
    const labels=[orientationEls.L,orientationEls.R,orientationEls.A,orientationEls.P,orientationEls.S,orientationEls.I];
    const placed=[];
    labels.forEach((el,i)=>{
      const dx=Number(el.getAttribute('x'))-64,dy=Number(el.getAttribute('y'))-64;
      const angle=Math.hypot(dx,dy)>3?Math.atan2(dy,dx):(-Math.PI/2+i*Math.PI/3);
      let best=null;
      for(const radius of [49,37,25])for(let step=0;step<24;step++){
        const turn=step===0?0:Math.ceil(step/2)*(step%2?1:-1)*Math.PI/12;
        const x=64+radius*Math.cos(angle+turn),y=64+radius*Math.sin(angle+turn);
        if(placed.some(p=>Math.abs(p.x-x)<22&&Math.abs(p.y-y)<15))continue;
        const score=Math.abs(turn)*20+Math.abs(radius-49);
        if(!best||score<best.score)best={x,y,score};
      }
      if(best){setLabel(el,best.x,best.y);placed.push(best);}
    });
"""
html=html.replace(anchor,layout+anchor,1)
old="ctx.fillText(label.textContent,Number(label.getAttribute('x')),Number(label.getAttribute('y')))"
new="ctx.strokeStyle='#080d12';ctx.lineWidth=3;ctx.lineJoin='round';ctx.strokeText(label.textContent,Number(label.getAttribute('x')),Number(label.getAttribute('y')));ctx.fillText(label.textContent,Number(label.getAttribute('x')),Number(label.getAttribute('y')))"
assert html.count(old)==1
html=html.replace(old,new,1)
viewer.write_text(html)
module=re.search(r'<script type="module">([\s\S]*?)</script>',html).group(1)
r=subprocess.run(['node','--input-type=module','--check'],input=module,text=True,capture_output=True)
assert r.returncode==0,r.stderr
print('Unified cyan/amber/lavender compass; collision-aware labels shared by HTML and canvas exports.')


Unified cyan/amber/lavender compass; collision-aware labels shared by HTML and canvas exports.


## Fully offline Full HTML
Embed the pinned Three.js 0.180.0 bundle (including OrbitControls and OBJLoader) and the public mouse-body OBJ. The GIF encoder, lung geometry and E02 ROI are already embedded. Source and homepage links remain optional hyperlinks. Rebuild Lite from this notebook before this final cell to keep the public preview small.

In [10]:
from pathlib import Path
import re, json
root=Path.cwd();viewer=root/'viewer/mlung3d_v3.3_full.html'
html=viewer.read_text()
engine=(root/'assets/three-offline-v0.180.0.js').read_text()
body=(root/'data/m07/mouse_reduced_face_7200.obj').read_text()
html=re.sub(r'<script type="importmap">.*?</script>',lambda m:'<script>'+engine.replace('</script','<\\/script')+'</script>',html,count=1)
html=html.replace("const THREE=await import('three');",'const {THREE,OrbitControls,OBJLoader}=MlungEngine;',1)
html=html.replace("  const {OrbitControls}=await import('three/addons/controls/OrbitControls.js');",'',1)
html=html.replace("  const {OBJLoader}=await import('three/addons/loaders/OBJLoader.js');",'',1)
html=re.sub(r"  const mouseObjUrl='[^']+';",lambda m:'  const mouseObjText='+json.dumps(body)+';',html,count=1)
html=html.replace("const text=await fetch(mouseObjUrl).then(r=>{if(!r.ok) throw new Error('mouse body download failed'); return r.text()});",'const text=mouseObjText;',1)
assert "await import('three" not in html and 'fetch(mouseObjUrl)' not in html
assert 'cdn.jsdelivr.net' not in html and 'raw.githubusercontent.com' not in html
html=html.replace("Use a current Chrome, Edge, Safari, or Firefox with internet access for Three.js.","Open this offline HTML in a current Chrome, Edge, Safari, or Firefox browser.")
viewer.write_text(html)
print('Offline Full HTML:',viewer.stat().st_size,'bytes')


Offline Full HTML: 46930443 bytes


## Panel typography and spacing
A final scoped stylesheet brings controls, Info and export dialogs to a consistent readable scale. Narrow control panels stack the numeric inputs beneath drag controls. Offline runtime and export canvas layouts are preserved.

In [11]:
from pathlib import Path
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html'
html=viewer.read_text()
css='<style id="v33PanelPolish">\n.panel{border-color:#2c404c;border-radius:18px;background:rgba(9,17,23,.96)}\n#ui{padding:20px!important;scrollbar-color:#406171 transparent}\n#ui h1{font-size:22px!important;line-height:1.25;margin-bottom:16px}\n#ui button{font-size:14px!important;min-height:44px;padding:10px 12px;border-radius:12px}\n#ui .homeButton{font-size:14px;min-height:44px;border-radius:12px}\n#ui .miniDetails{margin-top:12px;border-radius:14px}\n#ui .miniDetails summary{font-size:15px;font-weight:650;padding:14px}\n#ui .control label,#ui .control output,#ui .check,#ui .legendItem{font-size:14px}\n#ui .small,#ui #sideHint,#ui .planeAddressHelp{font-size:13px;line-height:1.6}\n#ui .planeControlHead{font-size:15px;gap:12px;flex-wrap:wrap}\n#ui .planeControlHead span{font-size:12px}\n#ui .tiltPadWrap b,#ui .offsetPadWrap b{font-size:14px}\n#ui .padNumberRow label{font-size:14px}\n#ui .padNumberShell input{font-size:16px}\n#ui .padStepArrows button{width:36px;height:26px;min-height:26px;padding:0;border-radius:0;font-size:16px!important}\n#ui textarea{font-size:13px!important;line-height:1.6!important}\n#infoPanel{box-sizing:border-box;width:min(540px,calc(100vw - 32px));padding:26px;z-index:70}\n#infoPanel h2{font-size:23px;line-height:1.3;padding-right:58px;margin-bottom:16px}\n#infoPanel .small,#infoPanel .citationList li{font-size:14px;line-height:1.65}\n#infoPanel .planeKicker{font-size:12px;letter-spacing:.06em}\n#infoPanel .planeCard{padding:16px;margin-top:16px;border-radius:12px}\n#infoPanel a{color:#9cdae9;text-decoration-color:#547b89;text-underline-offset:3px}\n#infoClose{font-size:13px;min-height:36px;padding:8px 12px}\n#pngSetup,#gifSetup,#videoSetup{box-sizing:border-box;width:min(420px,calc(100vw - 24px));padding:22px;font-size:15px;line-height:1.6}\n#pngSetup label,#gifSetup label,#videoSetup label{font-size:15px}\n#pngSetup select,#gifSetup select,#videoSetup select{font-size:15px;min-height:44px;padding:8px;width:100%;box-sizing:border-box}\n#pngSetup .small,#gifSetup .small,#videoSetup .small{font-size:13px;line-height:1.6}\n#pngSetup button,#gifSetup button,#videoSetup button{font-size:14px;min-height:42px}\n#orientation>div:first-child{font-size:15px!important;line-height:1.4}\n#orientationText{font-size:14px!important;line-height:1.6!important}.orientationLegend{font-size:13px;line-height:1.7}\n#alveolarDialog header h2{font-size:22px}#alveolarDialog .small{font-size:13px;line-height:1.6}\n#alveolarDialog button,#alveolarDialog select{font-size:14px;min-height:42px}\n@container(max-width:390px){#ui .tiltPadWrap,#ui .offsetPadWrap{grid-template-columns:1fr 1fr}#ui .padNumbers{grid-column:1/-1;grid-template-columns:1fr 1fr}#ui .offsetPadWrap .padNumbers{grid-template-columns:1fr}}\n@media(max-width:760px){#ui{padding:16px!important}#ui h1{font-size:20px!important}#infoPanel{width:calc(100vw - 20px);padding:20px}#ui .miniDetails summary{padding:12px}#presentationToolbar button{width:42px;height:42px;min-height:42px;min-width:42px}#alveolarDialog header{flex-wrap:wrap;gap:12px}}\n</style>'
assert 'id="v33PanelPolish"' not in html
html=html.replace('</head>',css+'</head>',1)
viewer.write_text(html)
print('Consistent panel typography and responsive controls applied.')


Consistent panel typography and responsive controls applied.


## Rotation speed and viewer reset
Offer 1, 1.5, 2, 2.5 and 3× rotation. The toolbar reset restores the paused default Ventral camera, initial plane values, default layer visibility and opacity, and closes auxiliary panels.

In [12]:
from pathlib import Path
import re, subprocess
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html';html=viewer.read_text()
assert html.count('const speedValues=[1,1.5,2]')==1
html=html.replace('const speedValues=[1,1.5,2]','const speedValues=[1,1.5,2,2.5,3]',1)
anchor='    <button id="rotationSpeed"'
button='<button id="resetViewer" type="button" aria-label="Reset viewer to default Ventral view" title="Reset viewer"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M20 7v5h-5M20 12a8 8 0 1 0-2.3 5.7"/></svg></button>\n'
assert html.count(anchor)==1
html=html.replace(anchor,button+anchor,1)
anchor='  const presetLabel='
reset="""  document.getElementById('resetViewer').addEventListener('click',()=>{
    stopAutoRotateByUser();speedIndex=0;controls.autoRotateSpeed=1;speedButton.textContent='1×';speedButton.setAttribute('aria-label','Rotation speed 1 times');
    mouseContextGroup.visible=false;applyLungPlacement(false);const bodyButton=document.getElementById('toggleMouseContext');bodyButton.classList.remove('active');bodyButton.textContent='Mouse body';
    clipping=false;sectionVisible=false;sectionDepth=0;sectionAngle=15;sectionTilt=0;sectionGroup.visible=false;
    const clipButton=document.getElementById('toggleClip');clipButton.classList.remove('active');clipButton.textContent='Clip';document.getElementById('toggleSection').textContent='Show plane';
    for(const id of ['depth','angle','angle2','opacity']){const input=document.getElementById(id);if(input){input.value=input.defaultValue;input.dispatchEvent(new Event('input',{bubbles:true}));}}
    for(const id of ['airwayVisible','lobesVisible','wholeAlveoli']){const input=document.getElementById(id);if(input){input.checked=input.defaultChecked;input.dispatchEvent(new Event('change',{bubbles:true}));}}
    updateSection();syncPlaneControls();document.getElementById('planeDetails').open=false;document.getElementById('viewDetails').open=true;
    for(const id of ['pngSetup','gifSetup','videoSetup','alveolarDialog','liteClipNotice']){const panel=document.getElementById(id);if(panel)panel.hidden=true;}
    const gate=document.getElementById('liteExportGate');if(gate)gate.open=false;
    setInfoOpen(false);currentView='Ventral';updateSideHint(currentView);camera.up.set(0,0,1);camera.position.copy(viewPosition('ventral'));controls.target.copy(activeCenter());camera.lookAt(activeCenter());controls.update();
  });
"""
assert html.count(anchor)==1
html=html.replace(anchor,reset+anchor,1)
viewer.write_text(html)
module=re.search(r'<script type="module">([\s\S]*?)</script>',html).group(1)
r=subprocess.run(['node','--input-type=module','--check'],input=module,text=True,capture_output=True)
assert r.returncode==0,r.stderr
print('Rotation speeds 1–3× and default Ventral reset added.')


Rotation speeds 1–3× and default Ventral reset added.


## Unclipped recording color key
Keep the accepted Clip layout. In the full-organ view, use a compact two-column color key at the lower right above the creator footer, opposite the orientation compass.

In [13]:
from pathlib import Path
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html';html=viewer.read_text()
html=html.replace('const x=clipped?Math.round(width*.63)+pad:width-width*.024-280*unit;', 'const x=clipped?Math.round(width*.63)+pad:width-width*.024-430*unit;',1)
html=html.replace('2*pad:280*unit;', '2*pad:430*unit;',1)
html=html.replace('const y=clipped?addressY-92*unit:height*.10;', 'const y=clipped?addressY-92*unit:height-150*unit;',1)
html=html.replace('const column=clipped?i%2:0,r=clipped?Math.floor(i/2):i;', 'const column=i%2,r=Math.floor(i/2);',1)
html=html.replace('(clipped?areaWidth/2:areaWidth)-20*unit', 'areaWidth/2-20*unit',1)
html=html.replace('y+(clipped?3:6)*row', 'y+3*row',1)
viewer.write_text(html)
print('Unclipped color key moved to lower right, above the creator footer.')


Unclipped color key moved to lower right, above the creator footer.


## Clear navigation labels
About & sources describes the information panel. Exploration stays separate from a compact row of GitHub and Author website links. Small outlined icons supplement the visible labels.

In [14]:
from pathlib import Path
import re
viewer=Path.cwd()/'viewer/mlung3d_v3.3_full.html';html=viewer.read_text()
html=html.replace('<button id="toggleInfo" class="ghost">Info</button>','<button id="toggleInfo" class="ghost" title="Model notes, sources, citation and acknowledgements">About &amp; sources</button>',1)
row=re.search(r'<div class="btnGrid two" aria-label="Explore and homepage">([\s\S]*?)</div>',html)
assert row
alveolar=re.search(r'<button id="openAlveolar"[^>]*>.*?</button>',row.group(1)).group(0)
new='<div class="btnGrid" aria-label="Explore and homepage">'+alveolar+'</div><nav class="projectLinks" aria-label="Project and author links"><a id="githubLink" href="https://github.com/arashabadi/mlung3d" target="_blank" rel="noopener noreferrer" aria-label="Open project source on GitHub"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M8 6 2 12l6 6M16 6l6 6-6 6M14 4l-4 16"/></svg>GitHub</a><a id="homeLink" href="https://arashabadi.github.io/" target="_blank" rel="noopener noreferrer" aria-label="Open author website"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M14 3h7v7M21 3l-11 11M10 5H3v16h16v-7"/></svg>Author website</a><a id="archiveLink" href="https://github.com/arashabadi/mlung3d/tree/main/viewer/archive" target="_blank" rel="noopener noreferrer">Archive</a></nav>'
html=html[:row.start()]+new+html[row.end():]
css='<style>#ui .projectLinks{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin:12px 0}#ui .projectLinks a{display:flex;align-items:center;justify-content:center;gap:8px;min-height:42px;padding:8px;border:1px solid #304b5a;border-radius:12px;color:#bddbe7;background:rgba(120,190,210,.04);font:600 14px system-ui,sans-serif;text-decoration:none}#ui .projectLinks a:hover{background:#19313e;border-color:#67c5e8}#ui .projectLinks a:focus-visible{outline:2px solid #67c5e8;outline-offset:3px}#ui .projectLinks svg{width:17px;height:17px;fill:none;stroke:currentColor;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round}#ui [aria-label="Explore and homepage"]{grid-template-columns:1fr}</style>'
html=html.replace('</head>',css+'</head>',1)
viewer.write_text(html)
assert html.count('id="githubLink"')==1 and html.count('id="homeLink"')==1
print('About & sources; separate GitHub and Author website links.')


About & sources; separate GitHub and Author website links.
